# Agent 运行循环：先猜，再运行

本练习不需要模型密钥，也不会调用真实大模型。ScriptedModel 按预先编排的步骤返回工具调用或交接请求，再观察 SDK 的处理；天气数据为固定值。

源码基线：openai-agents-python 0.24.0，commit 17d994bd3045dae71acda676a4ef4762ed8ecd94；Colab 安装发行包只固定版本，不能证明源码 commit 与本次 Cloud IDE 检出完全一致。

**Colab 使用**：如未安装 SDK，在新代码单元执行 `%pip install "openai-agents==0.24.0"`；必要时重启运行时并从头运行。不要填写密钥或内部数据。Cloud IDE 验证使用固定提交和专用 Python。

In [5]:
from importlib.metadata import version
import agents
print("openai-agents:", version("openai-agents"))
print("SDK 加载路径:", agents.__file__)
assert version("openai-agents") == "0.24.0", "请先安装 openai-agents==0.24.0"

ModuleNotFoundError: No module named 'agents'

## 1. 先猜：模型说要调用工具后，谁来执行？

选择你的答案：A. 模型直接运行本地函数；B. SDK 收到工具调用请求后运行本地函数。

接下来故意安排两个模型回复：第一个请求天气工具，第二个直接返回固定文字。请观察 SDK 在中间做了什么。

In [6]:
from agents import Agent, Handoff, RunConfig, RunHooks, Runner, function_tool
from agents.testing import ModelStep, ScriptedModel
from openai.types.responses import ResponseFunctionToolCall, ResponseOutputMessage
from openai.types.responses.response_output_text import ResponseOutputText

def tool_call(name, arguments, call_id="call_1"):
    return ResponseFunctionToolCall(id=call_id, call_id=call_id, type="function_call", name=name, arguments=arguments)

def final_message(text):
    return ResponseOutputMessage(id="msg_1", type="message", role="assistant", content=[ResponseOutputText(text=text, type="output_text", annotations=[], logprobs=[])], status="completed")

config = RunConfig(tracing_disabled=True)

ModuleNotFoundError: No module named 'agents'

In [ ]:
invocations = []
@function_tool
def get_weather(city: str) -> str:
    """Return illustrative weather for a city."""
    invocations.append(city)
    return f"{city}: sunny, 14-20C"

model = ScriptedModel([
    ModelStep(output=[tool_call("get_weather", '{"city":"Tokyo"}')]),
    ModelStep(output=[final_message("预设的最终回答")]),
])
agent = Agent(name="weather", tools=[get_weather], model=model)
result = await Runner.run(agent, "东京天气怎么样？", run_config=config)
print("模拟模型调用次数:", len(model.calls))
print("本地工具执行记录:", invocations)
print("第二次模型的输入:", model.calls[1].input)
print("最终输出:", result.final_output)
assert len(model.calls) == 2 and invocations == ["Tokyo"]

**观察**：答案是 B。第二次输入中会看到工具请求及对应结果，`call_id` 用于关联两者。最终回答是预设文字，不代表模型真的阅读天气并作答。

**源码定位**：[run_single_turn](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/run_loop.py#L2787) → [工具执行](https://github.com/openai/openai-agents-python/blob/17d994bd3045dae71acda676a4ef4762ed8ecd94/src/agents/run_internal/tool_execution.py#L2341)。

## 2. 先猜：传入错误类型，工具函数会执行吗？

这次模拟模型传入数字 `123`，但工具声明城市是字符串。先判断：

- A. 函数体会执行；
- B. 参数处理阶段失败，函数体不会执行。

再观察错误是如何进入下一次模型输入的。

In [ ]:
invocations.clear()
bad_model = ScriptedModel([
    ModelStep(output=[tool_call("get_weather", '{"city":123}')]),
    ModelStep(output=[final_message("预设的错误后回答")]),
])
bad_agent = Agent(name="weather", tools=[get_weather], model=bad_model)
bad_result = await Runner.run(bad_agent, "东京天气怎么样？", run_config=config)
outputs = [item.output for item in bad_result.new_items if item.type == "tool_call_output_item"]
print("工具函数执行记录:", invocations)
print("SDK 返回的工具输出:", outputs)
print("第二次模型的输入:", bad_model.calls[1].input)
assert invocations == [] and len(bad_model.calls) == 2 and len(outputs) == 1

**观察**：本次默认错误处理配置下，答案是 B，工具没有执行；SDK 生成通用错误消息并继续下一次模拟模型调用。修改错误处理配置后，不能假设结果相同。

## 3. 先猜：模型反复请求工具，会永远运行吗？

我们预设三次工具请求，并把 `max_turns` 设为 2。请预测：最终工具执行多少次？会得到普通最终回答，还是异常？

In [ ]:
from agents import MaxTurnsExceeded
invocations.clear()
repeat_model = ScriptedModel([ModelStep(output=[tool_call("get_weather", '{"city":"Tokyo"}', f"call_{n}")]) for n in (1, 2, 3)])
repeat_agent = Agent(name="weather", tools=[get_weather], model=repeat_model)
try:
    await Runner.run(repeat_agent, "东京天气怎么样？", max_turns=2, run_config=config)
    raise AssertionError("预期遇到轮次上限")
except MaxTurnsExceeded as exc:
    print("异常:", type(exc).__name__, str(exc))
print("模拟模型调用次数:", len(repeat_model.calls))
print("工具执行次数:", len(invocations))
assert len(repeat_model.calls) == 2 and len(invocations) == 2

**观察**：本次两轮中工具执行两次，尝试继续时触发上限异常。上限不会撤销已执行工具的副作用；这里没有模拟自定义上限错误处理器。

## 4. 先猜：handoff 以后谁继续？

预设 triage 输出交接给 specialist 的请求。它执行完以后，最终 Agent 是 triage 还是 specialist？

In [ ]:
class RecordHandoff(RunHooks):
    def __init__(self):
        self.events = []
    async def on_handoff(self, context, from_agent, to_agent):
        self.events.append(f"{from_agent.name} -> {to_agent.name}")

specialist_model = ScriptedModel([ModelStep(output=[final_message("预设的专家回答")])])
specialist = Agent(name="specialist", model=specialist_model)
transfer_name = Handoff.default_tool_name(specialist)
triage_model = ScriptedModel([ModelStep(output=[tool_call(transfer_name, "{}", "handoff_1")])])
triage = Agent(name="triage", model=triage_model, handoffs=[specialist])
recorder = RecordHandoff()
handoff_result = await Runner.run(triage, "请交接给专家", hooks=recorder, run_config=config)
print("交接请求名:", transfer_name)
print("交接事件:", recorder.events)
print("最终 Agent:", handoff_result.last_agent.name)
assert recorder.events == ["triage -> specialist"] and handoff_result.last_agent.name == "specialist"

**观察**：在这个预设交接路径中，SDK 切换到 specialist，随后运行 specialist 的模拟模型。与 Agent as tool 的区别是：交接后当前 Agent 发生切换；工具模式下主 Agent 可以拿回子 Agent 的工具结果继续处理。

**课后挑战**：把错误参数改成缺少 city，预测工具是否执行，再运行并记录结果。观察的是 SDK 如何处理预设请求，不是模型能否自行生成这个请求。

源码与实际日志见同目录的分享 Markdown；以上断言通过意味着这些指定路径符合本练习的期望，并不代表全部功能都被验证。